## What is a FiftyOne Dataset?

A **Dataset** is an ordered collection of **Samples**, each representing one piece of media (image, video, point cloud, etc.). 

Samples have **Fields** that store filepaths, labels, metadata, and any custom data you add. FiftyOne stores only labels and metadata in its database.

A Dataset lets you load, modify, visualize, evaluate, search, and sort your data and labels in a consistent format — all connected to the [FiftyOne App](https://docs.voxel51.com/cheat_sheets/fiftyone_terminology.html#data-model) for interactive exploration.

## Loading data into FiftyOne

FiftyOne's core data structure is the [`Dataset`](https://docs.voxel51.com/user_guide/basics.html#datasets), which holds an ordered collection of `Sample` objects. Each sample stores a **filepath** to the media (not the raw media itself), along with any labels, metadata, and custom fields. FiftyOne uses a lightweight non-relational database under the hood, so it scales to any dataset size without RAM concerns. 

[Learn more in these docs](https://docs.voxel51.com/user_guide/basics.html#datasets)


### Loading Raw Media

If you just have a folder of images or videos and no labels yet, use the [factory methods](https://docs.voxel51.com/user_guide/import_datasets.html#loading-media)

```python

import fiftyone as fo

# From a list of images
dataset = fo.Dataset.from_images(["/path/to/image1.jpg", "/path/to/image2.jpg"])

# From a directory of images
dataset = fo.Dataset.from_images_dir("/path/to/images")

# From a glob pattern
dataset = fo.Dataset.from_images_patt("/path/to/images/*.jpg")

# Same pattern works for videos
dataset = fo.Dataset.from_videos_dir("/path/to/videos")
```

### Loading Common Labeled Formats

If your data is in a standard format [COCO, VOC, CVAT, ImageClassificationDirectoryTree, etc.](https://docs.voxel51.com/user_guide/import_datasets.html#common-formats), use `Dataset.from_dir()`:

```python
import fiftyone as fo

# Load a COCO-format dataset
dataset = fo.Dataset.from_dir(
    dataset_dir="/path/to/dataset",
    dataset_type=fo.types.COCODetectionDataset,
)

# Or specify media and labels paths separately
dataset = fo.Dataset.from_dir(
    dataset_type=fo.types.COCODetectionDataset,
    data_path="/path/to/images",
    labels_path="/path/to/coco-labels.json",
)

# Import a random subset of 10 samples
dataset = fo.Dataset.from_dir(
    dataset_dir="/path/to/dataset",
    dataset_type=fo.types.COCODetectionDataset,
    max_samples=10,
    shuffle=True,
    seed=51,
)
```

### Loading Custom Formats

For data in a [custom format](https://docs.voxel51.com/user_guide/import_datasets.html#custom-formats), iterate over your files, create `Sample` objects, and add them to a dataset: 

```python
import glob
import fiftyone as fo

annotations = {"/path/to/images/000001.jpg": "dog", ...}

samples = []
for filepath in glob.glob("/path/to/images/*"):
    sample = fo.Sample(filepath=filepath)
    sample["ground_truth"] = fo.Classification(label=annotations[filepath])
    samples.append(sample)

dataset = fo.Dataset("my-classification-dataset")
dataset.add_samples(samples)
```

##### Your images are not duplicated

> Note that FiftyOne only stores the **filepath** to each image, not the raw image data itself. So, no duplicate copies of your files are created.

### Dataset persistence

The [`persistent` parameter](https://docs.voxel51.com/user_guide/using_datasets.html#dataset-persistence) controls whether a dataset survives beyond your Python session. By default (`persistent=False`), datasets are deleted when all FiftyOne processes exit. Set `persistent=True` to keep them permanently.

> **Note:** FiftyOne never touches your raw media files — only labels in the database are affected by deletion.

#### To make a dataset persist across sessions, you can either:

1. Set `persistent=True` at creation time:
   ```python
   import fiftyone as fo
   dataset = fo.Dataset("my_dataset", persistent=True)
   ```

2. Or set it after creation:
   ```python
   dataset.persistent = True
   ```

Once persistent, the dataset will remain available in the database even after you exit Python and start a new session.

#### Let's create a FiftyOne Dataset from our images

In [1]:
import fiftyone as fo 

dataset = fo.load_dataset("birds_nest")

/home/harpreet/miniconda/envs/fiftyone/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/harpreet/miniconda/envs/fiftyone/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [ ]:
import fiftyone as fo

SAMPLE_IMAGES_PATH = "bird_nest_data/20260400_bird_nest_voxel51/bird_nest_samples"

dataset = fo.Dataset.from_images_dir(
    images_dir=SAMPLE_IMAGES_PATH,
    name="birds__nest",
    persistent=True,
    overwrite=True
)

### Sample Tags in FiftyOne

We'll also add some tags to the Samples we just added.

In [ ]:
dataset.tag_samples("confirmed_samples")

dataset.create_index("tags")

### Sample tags

Every sample has [a `tags` field](https://docs.voxel51.com/user_guide/basics.html#tags) — a list of arbitrary strings, empty by default. 

Use them to mark splits (`"train"`, `"test"`), flag issues (`"low_quality"`), or encode any custom grouping you need to filter on later.

### Filtering by tags

Once samples are tagged, you can [easily create views](https://docs.voxel51.com/user_guide/using_views.html) based on those tags:

```python

# Only samples with the "validation" tag
val_view = dataset.match_tags("validation")

# Union of "validation" and "test"
val_test_view = dataset.match_tags(["validation", "test"])

# Samples that do NOT have the "test" tag
non_test_view = dataset.match_tags("test", bool=False)

# Samples that have BOTH "train" and "test" tags
both_view = dataset.match_tags(["train", "test"], all=True)
```
> **Note:** Sample tags are distinct from **label tags**, which are tags attached to individual `Label` instances (e.g., a specific detection) rather than the whole sample.

### Now let's add the remaining images to our existing Dataset

In [ ]:
ADDITIONAL_IMAGE_PATHS = [
    "bird_nest_data/20260400_bird_nest_voxel51/bucket_000", 
    "bird_nest_data/20260400_bird_nest_voxel51/bucket_001",
    "bird_nest_data/20260400_bird_nest_voxel51/bucket_002",
    "bird_nest_data/20260400_bird_nest_voxel51/bucket_003"
]

for path in ADDITIONAL_IMAGE_PATHS:
    dataset.add_images_dir(path)

### What's in a Dataset

When you [load a dataset](https://docs.voxel51.com/user_guide/basics.html#datasets) with images and then call `dataset`, FiftyOne displays a summary of the dataset's structure, including:

- **Name** of the dataset

- **Media type** (e.g., `image`)

- **Number of samples**

- **Persistence** status

- **Info** dictionary

- **Tags**

- **Sample fields** (e.g., `filepath`, `tags`, `metadata`, `ground_truth`, etc.)

#### Let's call our Dataset:


In [3]:
dataset

Name:        birds_nest
Media type:  image
Num samples: 17568
Persistent:  True
Tags:        []
Sample fields:
    id:                              fiftyone.core.fields.ObjectIdField
    filepath:                        fiftyone.core.fields.StringField
    tags:                            fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                        fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                      fiftyone.core.fields.DateTimeField
    last_modified_at:                fiftyone.core.fields.DateTimeField
    clip_embeddings:                 fiftyone.core.fields.VectorField
    qwen_embeddings:                 fiftyone.core.fields.VectorField
    tower_component_classifications: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classifications)
    tower_component_raw:             fiftyone.core.fields.StringField
    environment_classifications:     fiftyone.core.fiel

#### What is a Sample in FiftyOne?

A [**Sample** is the atomic element](https://docs.voxel51.com/user_guide/basics.html#samples) of a FiftyOne `Dataset`, you can think of it as a single row in a table.

Every sample:

- **Must have a `filepath`** pointing to its source data on disk (an image, video, etc.)

- **Stores all information** related to that piece of data, including labels, metadata, and any custom fields you add

You can inspect an individual sample using `dataset.first()` to see what fields are stored for each image: 

In [4]:
dataset.first()

<Sample: {
    'id': '69d53dca1c350d33570a9a6a',
    'media_type': 'image',
    'filepath': '/home/harpreet/workspace/siemens_workshop/bird_nest_data/20260400_bird_nest_voxel51/bird_nest_samples/40019154_1659368556072069497_1000.jpg',
    'tags': ['confirmed_samples'],
    'metadata': <ImageMetadata: {
        'size_bytes': 298289,
        'mime_type': 'image/jpeg',
        'width': 543,
        'height': 1026,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 4, 7, 17, 24, 26, 619000),
    'last_modified_at': datetime.datetime(2026, 4, 9, 15, 3, 24, 217000),
    'clip_embeddings': array([ 3.21720093e-02,  4.93851751e-02, -1.22511938e-01,  2.75244117e-02,
            3.98220234e-02,  5.58909476e-01,  4.29473698e-01,  3.85873675e-01,
            1.04926221e-01,  1.19225547e-01,  9.57623720e-02,  1.89939439e-02,
            8.59315991e-02,  2.24426225e-01, -8.29744101e-01, -1.37692988e-01,
           -1.24112320e+00, -1.16106585e-01, -1.83463544e-01, -6.0197800

### Default Fields

By default, every sample comes with these [fields](https://docs.voxel51.com/user_guide/using_datasets.html#fields):

| Field | Description |
|---|---|
| `id` | Unique ID, auto-generated when added to a dataset |
| `filepath` | Path to the source file on disk |
| `media_type` | Auto-computed from the filepath |
| `tags` | A list of string tags |
| `metadata` | Type-specific metadata (e.g., image dimensions) |
| `created_at` | When the sample was added to the dataset |
| `last_modified_at` | When the sample was last modified |


When [a new field is added to one sample]((https://docs.voxel51.com/user_guide/using_datasets.html#adding-fields-to-a-sample)) in a dataset, it becomes available on **all other samples** in that dataset (with a value of `None` if not set).

##### Notice that the `metadata` field is not populated for this dataset, let's go ahead and add that:

In [ ]:
dataset.compute_metadata()

### What `compute_metadata()` adds

Calling `dataset.compute_metadata()` [populates the `metadata` field](https://docs.voxel51.com/user_guide/basics.html#metadata) on each sample with type-specific information about the source media file.

**For Image Samples (`ImageMetadata`):**

| Field | Description |
|---|---|
| `size_bytes` | File size in bytes |
| `mime_type` | MIME type (e.g., `image/png`) |
| `width` | Image width in pixels |
| `height` | Image height in pixels |
| `num_channels` | Number of color channels |

#### We can inspect the first sample again to see what was populated:

In [5]:
dataset.first()['metadata']

<ImageMetadata: {
    'size_bytes': 298289,
    'mime_type': 'image/jpeg',
    'width': 543,
    'height': 1026,
    'num_channels': 3,
}>

## Inspect the Dataset in the App

Now that we have loaded the dataset, let's inspect the samples in [the App](https://docs.voxel51.com/user_guide/app.html).

In [ ]:
session = fo.launch_app(dataset, auto=False)
session.url

#### Work with a small sample of the Dataset

Since you're running this code live with me in this workshop, you may want to take a small sample of the Dataset, called a [Dataset View](https://docs.voxel51.com/user_guide/using_views.html).

1. **Confirmed Samples**: Uses `match_tags("confirmed_samples")` to [select all data points that have been tagged](https://docs.voxel51.com/cheat_sheets/filtering_cheat_sheet.html) as "confirmed_samples" - likely a curated set of validated examples.

2. **Remaining Samples**: Uses `match_tags("confirmed_samples", bool=False)` to select samples that are **NOT** tagged as "validation", then takes only 100 of them. This gives you a manageable sample from the remaining data.

3. **Combine**: The `concat()` method [merges these two views](https://docs.voxel51.com/cheat_sheets/views_cheat_sheet.html) into a single working dataset.



```python

# Take samples that MATCH the tag
matched_view = dataset.match_tags("confirmed_samples")

# Take samples that DO NOT match the tag
unmatched_view = dataset.match_tags("confirmed_samples", bool=False).shuffle(seed=51).take(100)

# Combine the two views
dataset = matched_view.concat(unmatched_view)

```

## FiftyOne Model Zoo

The [FiftyOne Model Zoo](https://docs.voxel51.com/model_zoo/overview.html) provides a powerful interface for downloading pre-trained models and applying them to your FiftyOne datasets.

### Basic Workflow

The [basic recipe](https://docs.voxel51.com/model_zoo/overview.html#basic-recipe) for working with zoo models looks like this: 

```python

import fiftyone as fo
import fiftyone.zoo as foz

# List available models
print(foz.list_zoo_models())

# Download and load a model
model = foz.load_zoo_model("faster-rcnn-resnet50-fpn-coco-torch")

# Run inference to a dataset
dataset.apply_model(model, label_field="predictions", confidence_thresh=0.5)

# Visualize in the App
session = fo.launch_app(dataset)
```

### Embeddings Support

Many models expose embeddings, which you can compute directly. 

We'll use CLIP to extract embeddings for the images in our dataset. 

First, instantiate the model:

In [ ]:
import fiftyone.zoo as foz 

clip_emb_model = foz.load_zoo_model("clip-vit-base32-torch")

## Computing embeddings on your Dataset

`compute_embeddings()` is a method on any FiftyOne `SampleCollection` (dataset or view) that generates vector embeddings for your samples using a model that exposes embeddings. 

### Patch Embeddings

You can also compute embeddings for **image patches** (e.g., bounding boxes from a detection field) using the companion method `compute_patch_embeddings()`:

```python

dataset.compute_patch_embeddings(
    model,
    patches_field="ground_truth",  # must be Detections, Polylines, etc.
    embeddings_field="patch_embeddings"
)
```

### Let's compute embeddings for our Dataset using CLIP

Once stored, these embeddings can be passed directly to [Brain methods](https://docs.voxel51.com/brain.html#similarity) like `compute_visualization()` or `compute_similarity()` via the `embeddings` parameter.

**Key Parameters:**

| Parameter | Default | Description |
|---|---|---|
| `embeddings_field` | `None` | Field name to store embeddings; if `None`, returned in-memory |
| `batch_size` | `None` | Batch size for models that support batching |
| `num_workers` | `None` | DataLoader workers (Torch models only) |
| `skip_failures` | `True` | Gracefully skip samples that fail |
| `progress` | `None` | Whether to show a progress bar |


In [ ]:
# Compute embeddings
dataset.compute_embeddings(
    model="clip-vit-base32-torch",
    embeddings_field="clip_embeddings",
    batch_size=128,
    num_workers=8
)

## FiftyOne Brain

The [FiftyOne Brain](https://docs.voxel51.com/brain.html) is a set of routines for dataset curation, analysis, and visualization. It provides methods to help you understand and improve your data and models, including:

- **`compute_visualization()`** – Visualize data in a low-dimensional embedding space

- **`compute_similarity()`** – Index data by similarity for querying

- **`compute_uniqueness()`** – Score how unique each sample is

- **`compute_mistakenness()`** – Identify [likely annotation errors](https://docs.voxel51.com/brain.html#label-mistakes)

- **`compute_hardness()`** – Score [how difficult each sample is](https://docs.voxel51.com/brain.html#sample-hardness) for a model

- **`compute_near_duplicates()` / `compute_exact_duplicates()`** – Find [near duplicate data](https://docs.voxel51.com/brain.html#near-duplicates) or [exact duplicate data](https://docs.voxel51.com/brain.html#exact-duplicates)


Most embedding-based Brain methods work out of the box with a default model. No configuration required.


## Computing Visualization for Embeddings

`fob.compute_visualization()` is a FiftyOne Brain method that [computes a low-dimensional representation](https://docs.voxel51.com/brain.html#visualizing-embeddings) of your samples or object patches, enabling interactive visual exploration of your data.


### Basic Usage

**Key Parameters**

| Parameter | Default | Description |
|---|---|---|
| `patches_field` | `None` | Field defining image patches to embed (Detections, Polylines, etc.) |
| `embeddings` | `None` | Pre-computed embeddings or field name |
| `model` | `None` | Model to use for generating embeddings |
| `method` | `"umap"` | Dimensionality reduction technique |
| `num_dims` | `2` | Number of output dimensions |
| `brain_key` | `None` | Key to save results under for later retrieval |
| `create_index` | `False` | Create a spatial index for efficient lasso queries |


In [ ]:
import fiftyone.brain as fob

# Create UMAP visualization
results = fob.compute_visualization(
    dataset,
    method="umap",  # Also supports "tsne", "pca"
    brain_key="clip_viz",
    embeddings="clip_embeddings",
    create_index=True
)

## Building a similarity index

[`compute_similarity()`](https://docs.voxel51.com/brain.html#similarity) indexes your images or patches by their embeddings, enabling three capabilities: 

- Sorting by visual similarity to selected samples

- Searching by natural language (with multimodal models like CLIP)

- Detecting duplicates.

### Text Similarity

If you use a multimodal model like CLIP, you can also [search by natural language text queries](https://docs.voxel51.com/brain.html#similarity) directly in the App or programmatically.


In [ ]:
import fiftyone.brain as fob

clip_sim_index = fob.compute_similarity(
    dataset,
    embeddings_field="clip_embeddings",
    model="clip-vit-base32-torch",
    brain_key="clip_sim",
)

### Extending the Dataset Zoo with Remote Source Zoo Models

The machine learning landscape moves fast.

New vision models drop weekly, **sometimes daily**, and waiting on a library's release cycle to access them isn't practical. Your own organization likely has **proprietary models it wants to bring into the same workflow** too.

FiftyOne's **[remote source zoo model](https://docs.voxel51.com/model_zoo/remote.html#working-with-remotely-sourced-models)** pattern solves exactly this problem: it lets you *register, download, and apply models* whose definitions live in external GitHub repositories or public URLs, using the *exact same API* you'd use for any built-in zoo model.

This pattern means the community, and your own team, can **ship model integrations on your own timelines**. 

### Computing embeddings with Qwen3-VL-Embedding

Qwen3-VL-Embedding maps text, images, and video into a unified representation space.

It's a model like CLIP in that you can perform cross-modal retrieval and understanding. 

You can check out the model implementation in [this GitHub Repo](https://github.com/harpreetsahota204/qwen3vl_embeddings).

### Let's use Qwen3-VL-Embedding to compute embeddings for our dataset

This is a familar pattern, so I won't explain it again...

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz 

fo.config.requirement_error_level=1

# Register and download
foz.register_zoo_model_source(
    "github.com/harpreetsahota204/qwen3vl_embeddings",
    overwrite=True
)

# Load model and compute embeddings
qwen_emb_model = foz.load_zoo_model(
    "Qwen/Qwen3-VL-Embedding-2B",
    media_type="image"
    )

dataset.compute_embeddings(
    qwen_emb_model,
    embeddings_field="qwen_embeddings",
    batch_size=64,
    num_workers=4,
)

# Create UMAP visualization
results = fob.compute_visualization(
    dataset,
    method="umap",  # Also supports "tsne", "pca"
    brain_key="qwen_viz",
    embeddings="qwen_embeddings",
)

qwen_sim_index = fob.compute_similarity(
    dataset,
    embeddings_field="qwen_embeddings",
    model="Qwen/Qwen3-VL-Embedding-2B",
    brain_key="qwen_sim",
)

##### Let's delete the model from GPU and clear cache

In [ ]:
import torch 

torch.cuda.empty_cache()
del qwen_emb_model

## Explore the embeddings panel

Let's now turn back to the App and explore the visualization in the [Embeddings Panel](https://docs.voxel51.com/user_guide/app.html#embeddings-panel).

I'll also show you how we can sort images by similarity and perform natural language search over our images in the App.

## Narrowing Down the Candidate Pool

We have embeddings, similarity search, and a ranked list of candidates — but a ranked list isn't a training set. 

Three problems remain: 

1. **Redundancy** 

2. Identifying **typical** samples

3. Identifying **unusual** samples.

### Redundancy

These images come from drone flights that capture many frames of the same tower from slightly different angles. 

Similarity search ranks all of them highly, but labeling the 12th photo of the same structure adds no training value.

[`compute_near_duplicates`](https://docs.voxel51.com/brain.html#near-duplicates) finds samples that are close enough in embedding space to be redundant for training. 

It computes nearest-neighbor distances and flags pairs below a configurable threshold as near-duplicates, giving you a `duplicates_view()` for inspection and `duplicate_ids` for removal.

#### Let's use `compute_near_duplicates` for our dataset:

In [6]:
import fiftyone.brain as fob

THRESHOLD = 0.10

duplicates_index = fob.compute_near_duplicates(
    dataset,
    embeddings="qwen_embeddings",
    threshold=THRESHOLD,  # tune for your model/data
)

Computing duplicate samples...
Duplicates computation complete


#### You can save a view of the duplicates and visualize in the App

In [ ]:
dups_view = duplicates_index.duplicates_view()

print(f"Identified {len(dups_view)} near duplicates at threshold of {THRESHOLD}")

dataset.save_view("dups_view", dups_view, overwrite=True)

Identified 1643 near duplicates at threshold of 0.1


#### Deduplicating the Dataset

- When FiftyOne builds a duplicate index, it groups near-duplicates together. Within each group, one sample is designated as the **representative** (the "canonical" one), and the rest are considered **duplicates**.

- `duplicates_index.duplicate_ids` returns only the IDs of the non-representative samples — i.e., the ones flagged as redundant.

- The representative from each group is **not** included in `duplicate_ids`.

It will **keep one** sample from each duplicate group and only remove the other(s). It does **not** delete both members of a duplicate pair.

In [ ]:
dup_ids = duplicates_index.duplicate_ids  # all near-duplicate sample IDs

clean_view = dataset.exclude(dup_ids)  # only unique/representative samples

dataset.save_view("clean_view", clean_view, overwrite=True)

### You can also truly delete the near duplicates from the Dataset

`dataset.delete_samples(dup_ids)` **permanently removes** those samples from the underlying dataset, not just from a view.

1. Remove all the duplicate (non-representative) sample records from the FiftyOne dataset permanently.

2. Keep the representative from each group (same as `exclude`, since `dup_ids` doesn't include representatives).

3. Leave your actual image/video files on disk untouched.

#### Key differences from the `exclude()` approach:

| | `dataset.exclude(dup_ids)` | `dataset.delete_samples(dup_ids)` |
|---|---|---|
| **Effect** | Creates a filtered *view* — the samples still exist in the dataset | **Permanently deletes** the samples from the dataset |
| **Reversible?** | Yes — the view is just a query; original data is untouched | **No** — once deleted, those samples are gone (unless you have a backup or re-ingest) |
| **Saved views** | Other saved views can still reference those samples | Those samples vanish from **all** views |
| **Disk** | No files are deleted from disk (images/videos remain) | Sample *records* are removed from the FiftyOne database, but the actual media files on disk are **not** deleted |


I'd recommend starting with the view-based approach until you're satisfied with the results. But, I'll delete them anyway...

In [9]:
dataset.delete_samples(dup_ids) 

### What's Typical?

`compute_representativeness` scores how well each sample typifies the dataset, normalized to [0, 1]. 

It works by clustering the embeddings and scoring each sample by its proximity to its cluster center — samples near the center of a dense cluster score highest.

High scorers are the common cases: the tower configurations, angles, and lighting that repeat throughout the data.


#### Let's use `compute_representativeness` for our dataset:

In [ ]:
import fiftyone.brain as fob

fob.compute_representativeness(
    dataset,
    embeddings="qwen_embeddings",
    representativeness_field="qwen_rep"
)

dataset.create_index("qwen_rep") # improved filtering in the App

Computing representativeness...
Computing clusters for 16467 embeddings; this may take awhile...
Representativeness computation complete


### What's Unusual?

`compute_uniqueness` answers the opposite question. 

It finds each sample's K nearest neighbors in embedding space and scores based on the weighted average distance to them, normalized to [0, 1]. The more isolated a sample, the higher the score.

High scorers are the edge cases — rare angles, unusual lighting, uncommon nest positions. 

Including these forces the model to handle more than just the common patterns.

#### Let's use `compute_uniqueness` for our dataset:

In [ ]:
import fiftyone.brain as fob

fob.compute_uniqueness(
    dataset,
    embeddings="qwen_embeddings",
    uniqueness_field="qwen_uniq"
)

dataset.create_index("qwen_uniq")

Computing uniqueness...
Uniqueness computation complete


#### Now let's go back to the App 

Let's visualize our embeddings and color by the scores we added to each sample and play around with the filters in the App sidebar.

## Similarity Search from Confirmed Examples

`sort_by_similarity` averages the embeddings of our confirmed nest images into a single "nest signature" and ranks every sample by distance to it. 

**Lower distance = more likely to contain a nest.**

It's a **heuristic** — the embedding captures the full scene, not just the nest.

But, the top candidates are significantly enriched **compared to random browsing**.

Pass in the confirmed sample IDs and every sample gets a `nest_similarity_dist` field. 

Sort by it in the App to browse from most to least likely.

#### Get the IDs of our confirmed nest examples


In [19]:
confirmed_ids = dataset.match_tags("confirmed_samples").values("id")

### Sort the dataset by similarity to the confirmed examples

The mean of their embeddings is computed automatically. You can learn more about similarity [here](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.sort_by_similarity).

By the way, if you ever load the dataset in a new enviornment, you can look at [these docs for managing brain runs](https://docs.voxel51.com/brain.html#managing-brain-runs).

For example if you need to load the similarity index, you can run:

```python

qwen_sim_index = dataset.load_brain_results("qwen_sim")
```

#### Now, let's sort by similarity

In [ ]:
sim_view = qwen_sim_index.sort_by_similarity(
    confirmed_ids,
    k=len(dataset),
    dist_field="nest_similarity_dist",
)

dataset.create_index("nest_similarity_dist")

## Finding nests

We now have similarity distances, uniqueness scores, and representativeness scores on every sample. 

No single field says "this is a nest," but together they let us hunt efficiently.

### Strategies to try in the App

- **Sort by `nest_similarity_dist` (ascending)** — browse images closest to your confirmed examples.

- **Embeddings Panel → "sort by similarity"** on a confirmed sample to see its visual neighbors. Spotting nests in a coherent group is faster than scanning randomly.

- **Sort by uniqueness (descending)** — nests are anomalies and may surface among outliers.

- **Natural language search** — try "bird nest on power line" or "nest on transmission tower."

- **Combine filters** — e.g., low `nest_similarity_dist` + high uniqueness to get a small, high-signal set.

Tag every sample with a nest as `confirmed_nest`. 

We'll use these in the next step.

In [3]:
## Tag what you find in the App
## Run this cell after you've finished your review pass

# How many new nests did we find?
new_nests = dataset.match_tags("confirmed_nest")
original_confirmed = dataset.match_tags("confirmed_samples")

print(f"Original confirmed examples: {len(original_confirmed)}")
print(f"Newly tagged nests: {len(new_nests)}")
print(f"Total confirmed nests: {len(original_confirmed) + len(new_nests)}")

Original confirmed examples: 10
Newly tagged nests: 52
Total confirmed nests: 62


## Per-sample similarity

Our first search averaged all confirmed examples into a single centroid embedding. 

That works when examples are homogeneous, but our nests span different scene types: tower crowns, crossarms, lattice structures.

The mean lands in a no-man's-land between them. 

More diversity makes it worse.

#### A Better approach

Search from each confirmed sample individually and combine the results.

Each sample searches its own neighborhood. 

A tower-crown nest finds more tower-crown images. 

A crossarm nest finds more crossarm images. Nothing gets averaged out.

This works because the embeddings capture scene structure (tower type, background, angle) even though they don't detect nests directly. 


#### Gather all confirmed nest sample IDs (original  + any newly tagged)

In [2]:
all_confirmed_ids = list(
    set(dataset.match_tags("confirmed_samples").values("id")) | 
    set(dataset.match_tags("confirmed_nest").values("id"))
)

print(f"Searching from {len(all_confirmed_ids)} confirmed examples")

Searching from 87 confirmed examples


#### For each candidate, track the smallest distance to ANY confirmed sample.

A candidate might appear in the neighborhood of multiple confirmed samples. We keep the minimum because that represents the strongest match.

In [ ]:
min_distance_per_candidate = {}

# Query each confirmed sample individually so we get per-candidate distances
# to every confirmed sample (a single bulk query would only rank against the set).
for confirmed_id in all_confirmed_ids:
    neighbors = qwen_sim_index.sort_by_similarity(
        confirmed_id, 
        k=50, 
        dist_field="per_sample_sim_dist"
    )
    
    neighbor_ids = neighbors.values("id")
    neighbor_dists = neighbors.values("per_sample_sim_dist")
    
    for neighbor_id, distance in zip(neighbor_ids, neighbor_dists):
        # Skip confirmed samples — we don't need to score them
        if neighbor_id in all_confirmed_ids:
            continue
        
        # Keep the minimum distance across all confirmed samples
        if neighbor_id not in min_distance_per_candidate or distance < min_distance_per_candidate[neighbor_id]:
            min_distance_per_candidate[neighbor_id] = distance

# Note: each sort_by_similarity call above overwrote `per_sample_sim_dist` on the
# dataset with values from a single confirmed sample. This write restores the
# correctly-merged minimum across ALL confirmed samples so downstream sorting is valid.
dataset.set_values("per_sample_sim_dist", min_distance_per_candidate, key_field="id")
dataset.create_index("per_sample_sim_dist")  # speeds up later sort/filter queries on this field

candidate_ids = list(min_distance_per_candidate.keys())
dataset.select(candidate_ids).tag_samples("similarity_candidate")

print(f"Found {len(min_distance_per_candidate)} unique candidates")

Found 604 unique candidates


#### Rinse and repeat

Now, turn back to the app and repeat the process

## Building the Annotation Queue

We've confirmed some nest images, but none have bounding boxes yet. We don't need to annotate all of them... 

We need the right ones.

By computing representativeness and uniqueness on just the confirmed nest samples, we can prioritize:

- **Representativeness**: which nests are the most common scenarios? Annotate these first, they're the base cases.

- **Uniqueness**: which nests are unusual? Annotate these second, they prevent the model from being brittle.

In [ ]:
import fiftyone.brain as fob

# Scope to just the confirmed nest images
confirmed_view = dataset.match_tags(["confirmed_samples", "confirmed_nest"])

# Within this set: what's a "typical" nest?
fob.compute_representativeness(
    confirmed_view,
    embeddings="qwen_embeddings",
    representativeness_field="nest_rep",
)

# Within this set: what's an "unusual" nest?
fob.compute_uniqueness(
    confirmed_view,
    embeddings="qwen_embeddings",
    uniqueness_field="nest_uniq",
)

dataset.save_view("confirmed_view",confirmed_view, overwrite=True)

Computing representativeness...
Computing clusters for 236 embeddings; this may take awhile...
Representativeness computation complete
Computing uniqueness...
Uniqueness computation complete


### Tagging the Annotation Queue

Now we can sort our confirmed nest images into an annotation priority order. 

The goal is to get the most training value out of the fewest annotated images.

#### Core training samples

The most typical nest appearances.

Annotate these first to cover the base scenarios your detector needs to learn.

In [3]:
from fiftyone import ViewField as F

confirmed_view.match(
    F("nest_rep") > 0.7
).tag_samples("annotate_first")


### Edge cases

The most unusual nest appearances.

Rare angles, uncommon positions, tricky lighting. Annotate these second to make the model robust.

In [4]:
from fiftyone import ViewField as F

confirmed_view.match(
    F("nest_uniq") > 0.7
).tag_samples("annotate_second")

### Everything else

Moderate on both scores. Annotate if there's budget remaining.

Not the most typical, not the most unusual.

In [5]:
from fiftyone import ViewField as F

confirmed_view.match(
    (F("nest_rep") <= 0.7) & (F("nest_uniq") <= 0.7)
).tag_samples("annotate_if_budget")

### Recap

1. **Computed embeddings** to map every image into a shared vector space.

2. **Removed near-duplicates** to eliminate redundant drone frames.

3. **Explored the dataset** using similarity, uniqueness, representativeness, and the embeddings panel to navigate 16,000+ images efficiently.

4. **Found and tagged nest images** through interactive exploration — sorting, browsing neighborhoods, spotting anomalies.

5. **Iterated the search** using per-sample similarity from our growing confirmed set.

6. **Built an annotation queue** by scoping representativeness and uniqueness to just the confirmed nests — prioritizing which to annotate first.

We went from a small number of confirmed examples and ~18,000 unlabeled images to images with confirmed nests, prioritized and ready for annotation. 

No ground truth required to get started. 

The same workflow applies to any new class — swap the confirmed examples, re-run, done.

### Let's return to the App

Now, let's go back to the App.

I'll show you how we can perforn annotation workflows from within FiftyOne.

## Prepare Dataset for fine-tuning

#### Split Dataset

We can use a utility function in the FiftyOne SDK to generate a random partition of the samples in the collection according to the specified split fractions.

Note: These tags are written directly onto the underlying dataset (not just the view). 

In [4]:
import fiftyone as fo
import fiftyone.utils.random as four
from fiftyone import ViewField as F

# Only include samples with at least one detection in the `predictions` field
annotated_samples = dataset.match(F("gt_nests.detections").length() > 0)

four.random_split(annotated_samples, {"train": 0.7, "test": 0.15, "val": 0.15}, seed=51)

#### Checking for Leaky Splits

[`compute_leaky_splits()`](https://docs.voxel51.com/brain.html#leaky-splits) detects near-duplicate samples that appear across dataset splits (e.g., train and test), which inflate evaluation metrics. 

It embeds all samples and flags close neighbors that cross split boundaries.

Splits can be defined via tags, a field on each sample, or explicit views. 

The key parameter is **`threshold`** (default `0.2`, range `[0.1, 0.25]`) — lower catches fewer leaks, higher risks false positives.

The returned index gives you a `leaks_view()` for inspection, `no_leaks_view()` to get a clean split, and `tag_leaks()` to mark them. 

See the [leaky datasets blog post](https://voxel51.com/blog/on-leaky-datasets-and-a-clever-horse) for why this matters.

In [5]:
import fiftyone.brain as fob

split_tags = ["train", "test"]

index = fob.compute_leaky_splits(
    annotated_samples, 
    splits=split_tags,
    embeddings="qwen_embeddings",
    similarity_index="qwen_sim",
    threshold=0.125
    )

# Get a view of all detected leaks
leaks = index.leaks_view()

# Tag all leaks with a "leak" tag
index.tag_leaks()

# Get the test split without leaks
test_set = index.split_views["test"]

test_set_no_leaks = index.no_leaks_view(test_set)

Computing duplicate samples...
Duplicates computation complete


### Export to YOLO Format Dataset

For the sake of simplicity, we will just train a YOLO model from Ultralytics.

In [6]:
# Define your splits (assumes samples are tagged "train", "val", etc.)
import fiftyone as fo 

export_dir = "yolo-dataset"
label_field = "gt_nests"

splits = ["train", "val"]
classes = ["bird_nest"]

for split in splits:
    split_view = annotated_samples.match_tags(split)
    split_view.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=label_field,
        split=split,
        classes=classes,
        export_media="symlink",
    )

 100% |█████████████████| 354/354 [311.0ms elapsed, 0s remaining, 1.1K samples/s]       
Directory 'yolo-dataset' already exists; export will be merged with existing files
 100% |███████████████████| 76/76 [64.6ms elapsed, 0s remaining, 1.2K samples/s] 


### Fine-tune Model

- **`epochs=300`** — generous ceiling; early stopping decides the actual stop point

- **`patience=20`** — stops if val mAP doesn't improve for 20 epochs; last run's best was epoch 27 so this is plenty

- **`cos_lr=True`** — cosine LR decay; smoother convergence on small datasets

- **`imgsz=[1026, 543]`** — high-res input so tiny nests stay detectable; matches your source image aspect

- **`close_mosaic=5`** — disables mosaic for only the last 5 epochs (not 10), giving more epochs of mosaic regularization

- **`mixup=0.1`** — blends pairs of training images; regularizes the box regressor (fixes the `val/box_loss` drift)

- **`copy_paste=0.1`** — pastes nest crops onto other backgrounds; teaches the model nest appearance independent of clutter, reduces false positives

- **`scale=0.9`** — wider scale jitter (default 0.5); breaks the narrow 0.1–0.3 width distribution and improves generalization to different nest sizes

- **`box=5.0`** — lowers box-regression loss weight from default 7.5; directly prevents the regressor from overfitting coordinates on 400 images

In [9]:
from ultralytics import YOLO 

base_yolo_model = YOLO("yolo26s.pt")

results = base_yolo_model.train(
    data="yolo-dataset/dataset.yaml",
    epochs=300,
    patience=20,
    cos_lr=True,
    imgsz=[1026, 543],
    close_mosaic=5,
    seed=51,
    mixup=0.1,
    copy_paste=0.1,
    scale=0.9,
    box=5.0,
)

New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.11.14 torch-2.10.0+cu130 CUDA:0 (NVIDIA GB10, 124611MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=yolo-dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=[1026, 543], int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

COMET ERROR: Failed to create Comet experiment, reason: ValueError('Comet.ml requires an API key. Please provide as the first argument to Experiment(api_key) or as an environment variable named COMET_API_KEY ')


WARNING ⚠️ Comet installed but not initialized correctly, not logging this run. Comet.ml requires an API key. Please provide as the first argument to Experiment(api_key) or as an environment variable named COMET_API_KEY 
Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.mod

### Run Inference on the Test Set

Now that training is done, we load the best checkpoint (`best.pt`) — the weights from the epoch with the highest validation mAP, saved automatically by Ultralytics.

We isolate the held-out `test` split with [`match_tags`](https://docs.voxel51.com/cheat_sheets/filtering_cheat_sheet.html) so the model is scored on samples it never saw during training or validation.

[`apply_model`](https://docs.voxel51.com/user_guide/using_datasets.html#applying-models) runs the YOLO model over the view and writes predictions directly to each sample under the `pred_nest` field as `fo.Detections`, ready for visualization in the App and evaluation in the next step.

In [3]:
from ultralytics import YOLO 

WEIGHTS_PATH = "runs/detect/train/weights/best.pt"

ft_nest_model = YOLO(WEIGHTS_PATH)

test_set = annotated_samples.match_tags("test")

test_set.apply_model(
    ft_nest_model, 
    label_field="pred_nest"
    )

/home/harpreet/miniconda/envs/fiftyone/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


 100% |███████████████████| 75/75 [2.3s elapsed, 0s remaining, 43.0 samples/s]      


### Evaluate Predictions Against Ground Truth

With predictions now on every test sample, we use [`evaluate_detections`](https://docs.voxel51.com/user_guide/evaluation.html#detections) to compare `pred_nest` against the `gt_nests` labels.

Under the hood, FiftyOne matches predicted and ground-truth boxes by IoU and computes standard detection metrics — precision, recall, mAP, and per-sample TP/FP/FN counts.

The `eval_key="nest_eval"` argument persists these results directly onto each sample, so you can filter, sort, and visualize errors in the App (e.g. `F("nest_eval_fp") > 0` to find images with false positives) and call [`results.print_report()`](https://docs.voxel51.com/user_guide/evaluation.html#viewing-results) later for a summary.

In [4]:
import fiftyone as fo

# Evaluate predictions w.r.t. labels in ground truth field
test_set.evaluate_detections(
    "pred_nest", 
    gt_field="gt_nests", 
    eval_key="nest_eval"
)

Evaluating detections...
 100% |███████████████████| 75/75 [74.1ms elapsed, 0s remaining, 1.0K samples/s] 


#### Now, let's go back to the App

We can evaluate our detections in the App for an interactive experience.

### Bonus: Enriching the Dataset with Vision Language Models

At this point we have a prioritized annotation queue, but every sample is  still just an image with scores and tags. 

We don't have any structured information about what's actually in each image.

Some interesting labels to have would be:

- what part of the tower is visible

- what the surrounding environment looks like

- where a nest might be located.

This is where Vision Language Models become useful. 

### We'll use Qwen 3.5 VL to..

- **Classify the tower component** visible in each image (tower crown, crossarm with insulators, conductor wires, etc.)

- **Classify the surrounding environment** (agricultural field, forest, road, etc.)

- **Flag image quality edge cases** (underexposed, obstructed by vegetation, tower out of frame, etc.)

- **Predict bounding boxes** for bird nests where the model can identify them

Combined with the similarity, representativeness, and uniqueness scores we already have, this gives us a richly annotated dataset where we can slice the data from multiple angles without having manually labeled a single image.

### Register the Remote Zoo Model Source and Load the Model

You can find the source code for this remote source zoo model implementation [here](https://github.com/harpreetsahota204/qwen3_5_vl).

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

fo.config.requirement_error_level=1

# Register and download
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/qwen3_5_vl",
    overwrite=True
)

qwen35_model = foz.load_zoo_model(
    "Qwen/Qwen3.5-2B",
    media_type="image",
    operation="classify",
)

#### Prompt the model to identify tower components

In [ ]:
tower_component_prompt = """
    You are shown an ariel image of a power transmission line. 
    Your task is to identify the part of the power transmission line this image depicts. Based on what you see in this image, choose EXACTLY one of the following classification labels: \n
    full transmission tower \n
    top of transmission tower \n
    crossarm with insulators \n
    conductor wires between towers \n
    base of transmission tower \n
    tower junction or connection point \n
    damper or spacer on wire \n
    no utility pole visible or powerlines visible \n
    "Analyze the image and return classifications as a JSON array: "
    '[{"label": "class_name"}]. '
"""

qwen35_model.prompt = tower_component_prompt

dataset.apply_model(
    qwen35_model,    
    label_field="tower_component",
    batch_size=64,
    num_workers=4
)

#### Prompt the model to identify tower components

In [ ]:
environment_prompt = """
    You are shown an ariel image of a power transmission line. Your task is to identify the environment surrounding the utility pole. Based on what you see in this image, choose EXACTLY one of the following classification labels: \n
    agricultural field \n
    dense forest \n
    body of water \n
    urban enviornment \n
    residential neighborhood \n
    industrial area with buildings \n
    parking lot \n
    road \n
    bare dirt area \n
    solar panels \n
    railway tracks \n
    pylons \n
    arid/desert landscape \n
    hilly or mountainous terrain \n
    "Analyze the image and return classifications as a JSON array: "
    '[{"label": "class_name"}]. '
"""

qwen35_model.prompt = environment_prompt

dataset.apply_model(
    qwen35_model,    
    label_field="environment",
    batch_size=128,
    num_workers=8
)

#### Prompt the model to identify image quality

In [ ]:
image_quality_prompt = """
    You are shown an ariel image of a power transmission line. Your task is to help identify edge cases in image quality. Based on what you see in this image, choose EXACTLY one of the following classification labels: \n
    clear well lit image \n
    underexposed dark image \n
    overexposed washed out image \n
    tower obscured by vegetation \n
    tower partially out of frame \n
    blurry or out of focus image \n
    image mostly showing ground with little tower visible \n
    image mostly showing sky with little tower visible \n
    "Analyze the image and return classifications as a JSON array: "
    '[{"label": "class_name"}]. '
"""

qwen35_model.prompt = image_quality_prompt

dataset.apply_model(
    qwen35_model,    
    label_field="image_quality",
    batch_size=64,
    num_workers=4
)

#### Prompt the model to draw bounding boxes

In [ ]:
qwen35_model.operation = "detect"

qwen35_model.prompt = """
    You are shown an ariel image of a power transmission line.
    Your task is to determine if there is a bird nest in this image and provide the bounding box coordinates for it. 
    If there is a bird nest, return results as JSON array:
    '[{"bbox_2d": [xmin, ymin, xmax, ymax], "label": "bird_nest"}, ...].'
"""

dataset.apply_model(
    qwen35_model,    
    label_field="qwen_bn",
    batch_size=64,
    num_workers=4
)

#### Other Remote Source Zoo Models to check out

- Molmo Point

- Gemma 4

#### Bonus Bonus - 3D Scene Generation

You ask why, I ask...why not?

If you want to run this, follow along with the installation instructions [here](https://github.com/harpreetsahota204/apple_sharp).

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Register the model source
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/apple_sharp",
    overwrite=True
)

# Load and apply the model
threed_model = foz.load_zoo_model("Apple/SHARP")


   73.5Kb [4.2ms elapsed, ? remaining, 300.8Mb/s]   


### Use Just a Small Subset of the Data

In [7]:
test_set_cloned = test_set.clone()
test_set_cloned.name = "threed_bn"
test_set_cloned.persistent=True

### Apply the Model

In [ ]:
test_set_cloned.apply_model(
    threed_model,
    "threed_path",
    batch_size=16,
    num_workers=4
)

### Parse 3D Results as a Grouped Dataset

In [ ]:
import fiftyone as fo
import os
from pathlib import Path

# Get filepaths from your existing dataset
filepaths = test_set_cloned.values("filepath")

# Create a new grouped dataset
grouped_dataset = fo.Dataset("bn_sharp_results", overwrite=True, persistent=True)
grouped_dataset.add_group_field("group", default="rgb")

# Process each filepath and create the group structure
samples = []
for filepath in filepaths:
    path = Path(filepath)
    base_dir = path.parent
    base_name = path.stem

    # Create paths for each modality
    rgb_path = filepath  # Original RGB image
    threed_path = os.path.join(base_dir, f"{base_name}.fo3d")  # 3D scene

    # Create a group for these related samples
    group = fo.Group()

    # Create samples for each modality
    rgb_sample = fo.Sample(filepath=rgb_path, group=group.element("rgb"))
    threed_sample = fo.Sample(filepath=threed_path, group=group.element("threed"))

    samples.extend([rgb_sample, threed_sample])

# Add all samples to the dataset
grouped_dataset.add_samples(samples)
